# Kapitel 20.3 - Asyncio Grundlagen und Task-Management

Dieses Notebook behandelt Coroutines, Event Loop, Tasks und `gather`.

# Lernziele

- Coroutines korrekt definieren und starten
- `await`, `create_task`, `gather` sicher nutzen
- Timeouts und Cancellation verstehen

# Voraussetzungen

- Kapitel 20.1

# Theorie

Asyncio ist single-threaded Concurrency: waehrend eine Aufgabe wartet, kann die naechste laufen.
Ideal fuer viele gleichzeitige I/O-Operationen.

# Erklaerung

`async def` erstellt Coroutine-Funktionen.
`await` gibt Kontrolle an den Event Loop zurueck.
Tasks planen Coroutines zur gleichzeitigen Ausfuehrung.

# Syntax

```python
async def f():
    await asyncio.sleep(1)

task = asyncio.create_task(f())
await task
```

# Merke

Blockierende Funktionen wie `time.sleep` gehoeren nicht direkt in Async-Code.

# Parameter

- `asyncio.gather(*coros, return_exceptions=False)`
- `asyncio.wait_for(coro, timeout=...)`

# Rueckgabewert

`await` liefert den Rueckgabewert der Coroutine.
`gather` liefert eine Liste in derselben Reihenfolge wie uebergeben.

In [ ]:
# Beispiel 1
import asyncio

async def lade(name, sekunden):
    await asyncio.sleep(sekunden)
    return f'{name} fertig'

async def main():
    r1 = await lade('A', 1)
    r2 = await lade('B', 1)
    print(r1, r2)

asyncio.run(main())

In [ ]:
# Beispiel 2
async def main_parallel():
    ergebnisse = await asyncio.gather(lade('A', 1), lade('B', 1), lade('C', 1))
    print(ergebnisse)

asyncio.run(main_parallel())

In [ ]:
# Beispiel 3: Timeout
async def langsam():
    await asyncio.sleep(2)
    return 'ok'

async def timeout_demo():
    try:
        print(await asyncio.wait_for(langsam(), timeout=1))
    except asyncio.TimeoutError:
        print('Timeout aufgetreten')

asyncio.run(timeout_demo())

# Praxisbeispiel

Simuliere 20 API-Calls mit unterschiedlichen Antwortzeiten und sammle Resultate robust.

# Haeufige Fehler

1. Coroutine ohne `await` aufrufen.
2. `time.sleep` statt `asyncio.sleep` verwenden.
3. Exceptions in Tasks nicht behandeln.

# Best Practice

- Pro API-Aufruf Timeout setzen.
- Ergebnisse und Fehler separat erfassen.
- Event Loop nur einmal starten (`asyncio.run`).

# Tipp

Baue kleine Async-Hilfsfunktionen statt grosser Monolithen.

# Uebung

Erstelle 10 Tasks, davon 2 mit absichtlichem Fehler, und sammle alles mit `return_exceptions=True`.

# Loesung

Nutze `asyncio.gather(*tasks, return_exceptions=True)` und pruefe pro Eintrag, ob es eine Exception ist.

# Zusammenfassung

Du kannst jetzt Asyncio strukturiert einsetzen und mit Timeouts/Errors umgehen.

# Weiterfuehrende Links

- Python `asyncio` guide

## Technischer Tiefgang

Der Fokus liegt auf reproduzierbaren technischen Entscheidungen statt auf isolierten Einzelbeispielen.
Dabei werden Architektur, Robustheit und Betriebsfaehigkeit gemeinsam betrachtet.

## Zentrale Fachbegriffe

Race Condition
Critical Section
Event Loop
Backpressure
Cancellation
Timeout Budget

In [ ]:
# asyncio Timeout-Guard
import asyncio
async def slow_job():
    await asyncio.sleep(2)
    return "done"
async def main():
    try:
        print(await asyncio.wait_for(slow_job(), timeout=1.0))
    except asyncio.TimeoutError:
        print("timeout")
asyncio.run(main())

## Fallstudie (Praxis)

Waehle ein realistisches Produktionsszenario und beschreibe systematisch Ursache, Risiko und technische Gegenmassnahmen.
Ergaenze mindestens ein Kriterium fuer Monitoring und ein Kriterium fuer Release-Entscheidungen.

## Haeufige Fehler und Debugging-Checkliste

- Ist das Problem reproduzierbar mit klaren Schritten?
- Sind relevante Signale vorhanden (Logs, Tests, Metriken)?
- Wurde eine konkrete Hypothese getestet und falsifiziert/bestaetigt?
- Ist die Korrektur durch einen Regressionstest abgesichert?
- Wurden Betriebsfolgen und Dokumentation mit aktualisiert?

## Pruefungsfragen und Kurzloesungen

1. Warum ist Reproduzierbarkeit in Fehleranalyse und Betrieb zentral?
Kurzloesung: Ohne reproduzierbare Befunde sind Ursachenanalyse, Fix und Absicherung nicht belastbar.
2. Was unterscheidet technische Begriffe von bloessem Buzzword-Einsatz?
Kurzloesung: Praezise Begriffe steuern messbare Entscheidungen und verbessern Teamkommunikation.
3. Welche Mindestkriterien sollte ein Release-Gate enthalten?
Kurzloesung: Teststatus, Sicherheitschecks, Fehlerbudget und nachvollziehbare Freigabeentscheidung.